In [1]:
!ls ../data/wiki*.jsonl

../data/wikipedia_synthetic1.jsonl  ../data/wikipedia_synthetic5.jsonl
../data/wikipedia_synthetic2.jsonl  ../data/wikipedia_synthetic6.jsonl
../data/wikipedia_synthetic3.jsonl  ../data/wikipedia_synthetic7.jsonl
../data/wikipedia_synthetic4.jsonl  ../data/wikipedia_synthetic.jsonl


In [2]:
import json
import pandas as pd
from glob import glob

PATH = "../data/wiki*.jsonl"
files = glob(PATH)
# files = [f for f in files if "8" in f or "9" in f ]

def load_json(file):
    def get_data(raw):
        line = json.loads(raw)
        return {
            "text": line["text"],
            "labels": line.get("labels"),
            "not_labels": line.get("not_labels")
        }
    with open(file, "r") as f:
        data = [get_data(line) for line in f]
    return data

files_data = [load_json(file) for file in files]
df = pd.DataFrame([i for file_data in files_data for i in file_data])

In [3]:
df.sample(5)

,text,labels,not_labels
9556,This academic book cataloging Italian administ...,"[administrative_context, feature_enumeration, ...","[geographical_specificity, user_scenario_depic..."
19237,If you're planning to explore Lazio beyond the...,"[practical_travel_advice, geographic_orientati...","[family_heritage_connection, nostalgic_remembr..."
12785,The lead single 'Unbreakable' went straight to...,"[singles_performance_data, chart_performance_a...","[regional_market_variations, reissue_informati..."
9212,A retired professor returns to the college's h...,"[legacy_inheritance, temporal_continuity, ritu...","[academic_competition, individual_agency, yout..."
7350,One acronym. Four realities. A congressional d...,"[list_as_proof_device, authority_questioning, ...","[myth_deconstruction, narrative_reframing, epi..."


In [4]:
def merge_group(group):
    merged_labels = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels  # remove any intersection
    return pd.Series({
        "labels": sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
    })

df = (
    df.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"{len(df)} unique texts after merging")
df.sample(5)


21118 unique texts after merging


,text,labels,not_labels
5837,Margaret Bailes achieved Olympic gold medal st...,"[competitive achievement documentation, premat...","[institutional memory preservation, retrospect..."
1972,"The Langlands program has been called a ""grand...",[correspondence between number theory and harm...,[classification of representations via automor...
14620,The Board underscores the necessity of institu...,"[development_assistance_commitment, institutio...","[cross_border_collaboration, cultural_diversit..."
1831,"Mitochondria contain their own circular DNA, r...",[endosymbiotic origin of organelles],"[horizontal gene transfer mechanism, metabolic..."
14774,The carved pews once reserved for the Lord of ...,"[anti_patriarchal_statement, collective_empowe...","[architectural_symbolism, legacy_affirmation, ..."


In [5]:
import random
from collections import Counter, defaultdict

random.seed(42)
test_ratio = 0.1

# ── 1. Label frequency overview ───────────────────────────────────────────────
label_counts = Counter(lab for labs in df["labels"] for lab in labs)
not_label_counts = Counter(lab for labs in df["not_labels"] for lab in labs)

print(f"Unique positive labels : {len(label_counts)}")
print(f"Unique negative labels : {len(not_label_counts)}")
print(f"\nTop-10 positive labels:")
for lab, cnt in label_counts.most_common(10):
    print(f"  {lab:50s}  {cnt}")

# ── 2. Select held-out (test) labels ──────────────────────────────────────────
# Build label → row-index mapping
label_to_rows = defaultdict(set)
for i, labs in enumerate(df["labels"]):
    for lab in labs:
        label_to_rows[lab].add(i)

# Shuffle to avoid systematic bias, then greedily cover rows until target
all_labels = list(label_counts.keys())
random.shuffle(all_labels)

target_test_n = int(test_ratio * len(df))
test_labels = set()
test_row_indices = set()

for label in all_labels:
    if len(test_row_indices) >= target_test_n:
        break
    new_rows = label_to_rows[label] - test_row_indices
    if new_rows:
        test_labels.add(label)
        test_row_indices |= new_rows

train_labels = set(label_counts.keys()) - test_labels

print(f"\nLabel vocabulary split:")
print(f"  Train labels : {len(train_labels)}")
print(f"  Test labels  : {len(test_labels)}")

# ── 3. Assign rows ─────────────────────────────────────────────────────────────
# A row goes to test if ANY positive label is a held-out test label
is_test = df["labels"].apply(lambda labs: bool(set(labs) & test_labels))

df_train = df[~is_test].reset_index(drop=True)
df_test  = df[is_test].reset_index(drop=True)

print(f"\nRow split:")
print(f"  Train : {len(df_train):6d}  ({len(df_train) / len(df):.1%})")
print(f"  Test  : {len(df_test):6d}  ({len(df_test)  / len(df):.1%})")

# ── 4. Sanity check: zero label leakage ───────────────────────────────────────
train_positive_labels = set(lab for labs in df_train["labels"] for lab in labs)
leakage = test_labels & train_positive_labels
print(f"\nLabel leakage into train positives (must be 0): {len(leakage)}")


Unique positive labels : 46617
Unique negative labels : 37886

Top-10 positive labels:
  comparative_analysis                                127
  historical_documentation                            107
  heritage_preservation                               100
  legacy_preservation                                 87
  historical_context                                  85
  historical_continuity                               83
  institutional_critique                              83
  temporal_progression                                74
  institutional_affiliation                           71
  historical_significance                             70

Label vocabulary split:
  Train labels : 45246
  Test labels  : 1371

Row split:
  Train :  19004  (90.0%)
  Test  :   2114  (10.0%)

Label leakage into train positives (must be 0): 0


In [21]:
import pandas as pd
from IPython.display import display

train_pos = set(lab for labs in df_train["labels"]     for lab in labs)
train_neg = set(lab for labs in df_train["not_labels"] for lab in labs)
test_pos  = set(lab for labs in df_test["labels"]      for lab in labs)
test_neg  = set(lab for labs in df_test["not_labels"]  for lab in labs)

total = len(test_pos) + len(test_neg)

# For a given target set, split by how the label was seen in train
def bucket(s):
    return {
        "seen in train as positive only":          len((s & train_pos) - train_neg),
        "seen in train as negative only":          len((s & train_neg) - train_pos),
        "seen in train as both pos and neg":       len(s & train_pos & train_neg),
        "never seen in train":                     len(s - train_pos - train_neg),
    }

tp = bucket(test_pos)   # labels used as ground-truth positives in test
tn = bucket(test_neg)   # labels used as hard negatives in test
index = list(tp.keys())

counts = pd.DataFrame({
    "used as positive in test": [tp[k] for k in index],
    "used as negative in test": [tn[k] for k in index],
    "total":                    [tp[k] + tn[k] for k in index],
}, index=index)
counts.index.name = "how the label was seen in train"

ratios = (counts / total * 100).round(1).astype(str) + "%"

print(f"Unique test positive labels: {len(test_pos)}   "
      f"Unique test negative labels: {len(test_neg)}   "
      f"Total: {total}\n")
print("── Counts ──")
display(counts)
print("── % of total test labels ──")
display(ratios)


Unique test positive labels: 6670   Unique test negative labels: 6177   Total: 12847

── Counts ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,893,1606,2499
seen in train as negative only,2397,799,3196
seen in train as both pos and neg,1802,3161,4963
never seen in train,1578,611,2189


── % of total test labels ──


,used as positive in test,used as negative in test,total
how the label was seen in train,,,
seen in train as positive only,7.0%,12.5%,19.5%
seen in train as negative only,18.7%,6.2%,24.9%
seen in train as both pos and neg,14.0%,24.6%,38.6%
never seen in train,12.3%,4.8%,17.0%


In [7]:
import datasets

train_ds = datasets.Dataset.from_pandas(df_train)
test_ds = datasets.Dataset.from_pandas(df_test)

dataset = datasets.DatasetDict({
    "train": train_ds,
    "test": test_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 19004
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 2114
    })
})

In [7]:
dataset.push_to_hub("alexneakameni/ZSHOT-HARDSET-v2", commit_description="Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2/commit/1d8aedc4d03b6af64f0df48a85b8c3ac80f055a3', commit_message='Upload dataset', commit_description='Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.', oid='1d8aedc4d03b6af64f0df48a85b8c3ac80f055a3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-v2'), pr_revision=None, pr_num=None)